# WAV1 — Stage-2 isolated Kaggle arm
Attach the private core dataset and `af2-spectral-stage1-sequential-output.zip`. This notebook reconstructs the frozen Stage-1 decision first, verifies WAV1's own static authorization, then trains WAV1 only. Test remains locked.


In [ ]:
import importlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
ARM='WAV1'; WORK=Path('/kaggle/working'); INPUT=Path('/kaggle/input'); REPO=WORK/'coffee-bean-detection'; OUT=WORK/'af2-spectral-factorization-v1'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    result=subprocess.run(['git','clone','--depth','1','--branch','agent/af2-spectral-factorization','https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for module_name in list(sys.modules):
    if module_name=='coffee_detector' or module_name.startswith('coffee_detector.'): sys.modules.pop(module_name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input,restore_spectral_kaggle_run
from coffee_detector.af2_spectral.audit import run_spectral_static_audit
from coffee_detector.experiments.run_faruq_v3_af2_spectral_decision import run_spectral_decision
DATA,A,CONTRACT=prepare_af2_spectral_kaggle_input(INPUT,WORK); assert CONTRACT['decision']=='PASS' and CONTRACT['test_images_accessed'] is False
OUT.mkdir(exist_ok=True); REPORTS=OUT/'val_reports'; REPORTS.mkdir(parents=True,exist_ok=True)
stage1_zip=sorted(path for path in INPUT.rglob('af2-spectral-stage1-sequential-output.zip') if path.is_file())
if len(stage1_zip)!=1: raise FileNotFoundError(f'Harus ada tepat satu Stage1 ZIP; ditemukan {stage1_zip}')
stage1_root=WORK/'af2-spectral-stage1-input'
if stage1_root.exists(): shutil.rmtree(stage1_root)
stage1_root.mkdir(parents=True)
with zipfile.ZipFile(stage1_zip[0],'r') as handle: handle.extractall(stage1_root)
for arm in ('AF2WIN','AF2ORI','AF2POL','AF2SOFT','AF2LUM'):
    matches=sorted(path for path in stage1_root.rglob(f'{arm}_seed42_result.json') if path.is_file())
    if len(matches)!=1: raise FileNotFoundError(f'Hasil Stage1 {arm} harus tepat satu; ditemukan {matches}')
    shutil.copy2(matches[0],REPORTS/matches[0].name)
baseline=sorted(path for path in INPUT.rglob('lfdet_afab_seed42_screening.json') if path.is_file())
if len(baseline)!=1: raise FileNotFoundError(f'Evidence AF2 harus tepat satu; ditemukan {baseline}')
stage1=run_spectral_decision(OUT,baseline[0],stage='stage1'); assert stage1['next']=='AUTHORIZE_STAGE2' and stage1['test_opened'] is False
print('STAGE1 DECISION:',stage1['decision'],'retained=',stage1['retained'],'next=',stage1['next'])
STATIC=OUT/'static_audit.json'; audit=run_spectral_static_audit(A['D0_seed42_best.pt'],STATIC,device='cuda:0')
entry=audit['arms'][ARM]; print('STATIC',ARM,'failed_gates=',entry.get('failed_gates',[]),'authorized=',audit.get('arm_authorization',{}).get(ARM))
if not audit.get('arm_authorization',{}).get(ARM,False): raise RuntimeError(f'{ARM} static audit gagal: {entry.get("failed_gates",[])}')
audit['overall_decision']=audit.get('decision'); audit['scope']=f'stage2:{ARM}'; audit['scoped_arms']=[ARM]; audit['decision']='PASS'; audit['training_authorized']=True; audit['test_access_authorized']=False
STATIC.write_text(json.dumps(audit,indent=2)+'\n',encoding='utf-8')
CONFIG=REPO/f'configs/af2_spectral/{ARM}_yolo26n.yaml'; restore_spectral_kaggle_run(INPUT,OUT,arm=ARM,seed=42,d0_checkpoint=A['D0_seed42_best.pt'],config=CONFIG)
LOG=OUT/f'{ARM}_seed42_run.log'; RESULT=REPORTS/f'{ARM}_seed42_result.json'
CMD=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_spectral_arm','--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--d0-checkpoint',str(A['D0_seed42_best.pt']),'--static-audit',str(STATIC),'--output-root',str(OUT),'--device','0','--authorize-training']
if not RESULT.is_file():
    with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(CMD,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    seen=-1
    while process.poll() is None:
        csv=OUT/ARM/f'{ARM}_seed42'/'results.csv'; n=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if n!=seen: print(f'{ARM}: {n}/50 epoch | log={LOG}',flush=True); seen=n
        time.sleep(120)
    if process.returncode:
        tail='\n'.join(LOG.read_text(errors='replace').splitlines()[-180:])
        raise RuntimeError(f'{ARM} gagal: returncode={process.returncode}\n--- LOG TAIL ---\n{tail}')
assert RESULT.is_file(); payload=json.loads(RESULT.read_text(encoding='utf-8')); assert payload['evaluation_split']=='val' and payload['test_images_accessed'] is False
print(json.dumps(payload,indent=2))
archive=shutil.make_archive(f'/kaggle/working/{ARM}_seed42_output','zip',OUT)
print('DOWNLOAD SEBELUM STOP SESSION:',archive)
